In [9]:
%%capture
%pip install pandas requests

In [10]:
import pandas as pd
import requests
import json
import time

# Load your distinct materials CSV
df = pd.read_csv('materials_distinct.csv')

GETTY_SPARQL_URL = "https://vocab.getty.edu/sparql"

In [11]:
def fetch_aat_id(term):
    """
    Queries Getty SPARQL endpoint safely using regex filtering to avoid Lucene errors,
    and handles non-JSON error responses gracefully.
    """
    # Remove complex descriptions to catch the base material (e.g., "Adobe finish structure" -> "Adobe")
    # Take the first 2 words if it's a long description string to maximize hits
    search_term = " ".join(term.split()[:2]).replace('"', '\\"').strip()
    
    # A safer SPARQL query structure for the Getty Endpoint using case-insensitive regex
    query = f"""
    SELECT DISTINCT ?subject WHERE {{
      ?subject a skos:Concept ;
               xl:prefLabel/xl:literalForm ?label .
      FILTER(regex(str(?label), "^{search_term}$", "i"))
      FILTER regex(str(?subject), "http://vocab.getty.edu/aat/")
    }} LIMIT 1
    """
    
    try:
        response = requests.get(
            GETTY_SPARQL_URL, 
            params={'query': query, 'format': 'json'}, 
            headers={
                'User-Agent': 'StrapiImporter/1.0 (your-email@example.com)',
                'Accept': 'application/sparql-results+json'  # Explicitly tell the server we want JSON
            },
            timeout=15
        )
        
        # Check if the server actually threw an error page
        if response.status_code != 200:
            print(f"Server returned HTTP {response.status_code} for '{term}'")
            return None
            
        # Check if the response content is actually JSON
        if 'application/json' not in response.headers.get('Content-Type', '').lower() and \
           'sparql-results+json' not in response.headers.get('Content-Type', '').lower():
            print(f"Non-JSON response received for '{term}': HTML or Plain Text returned.")
            return None

        data = response.json()
        results = data.get('results', {}).get('bindings', [])
        if results:
            uri = results[0]['subject']['value']
            return uri.split('/')[-1]
            
    except requests.exceptions.JSONDecodeError:
        print(f"Failed to parse JSON for '{term}' (Server returned malformed data)")
    except Exception as e:
        print(f"Error fetching AAT for '{term}': {e}")
    
    return None

In [12]:
strapi_payloads = []

print(f"Starting AAT reconciliation for {len(df)} terms...")

for index, row in df.iterrows():
    en_name = str(row['material_en']).strip()
    ar_name = str(row['material_ar']).strip()
    note = str(row['review_note']) if pd.notna(row['review_note']) else ""
    
    print(f"[{index+1}/{len(df)}] Querying Getty for: {en_name}")
    
    # Automate role classification based on common support indicators
    support_keywords = ['canvas', 'paper', 'parchment', 'wood panel', 'tablet', 'fabric', 'linen']
    material_type = "support" if any(kw in en_name.lower() for kw in support_keywords) else "medium"
    
    # Fetch ID from Getty
    aat_id = fetch_aat_id(en_name)
    
    # Rate limit friendliness
    time.sleep(0.2)
    
    # Construct localized data entry mapping
    item_payload = {
        "type": material_type,
        "vocab": "AAT",
        "refid": aat_id if aat_id else "PENDING_MANUAL_REVIEW",
        "review_note": note,
        # Default/English translation entries
        "en_data": {
            "name": en_name
        },
        # Arabic translation entries
        "ar_data": {
            "name": ar_name
        }
    }
    strapi_payloads.append(item_payload)

# Save configuration to a structured JSON file ready for Strapi import
with open('strapi_ready_materials.json', 'w', encoding='utf-8') as f:
    json.dump(strapi_payloads, f, ensure_ascii=False, indent=2)

print("Reconciliation complete! Data stored in 'strapi_ready_materials.json'")

Starting AAT reconciliation for 240 terms...
[1/240] Querying Getty for: Adobe finish structure
Failed to parse JSON for 'Adobe finish structure' (Server returned malformed data)
[2/240] Querying Getty for: Agate
Failed to parse JSON for 'Agate' (Server returned malformed data)
[3/240] Querying Getty for: Alabaster
Failed to parse JSON for 'Alabaster' (Server returned malformed data)
[4/240] Querying Getty for: Aluminosilicate glass folios
Failed to parse JSON for 'Aluminosilicate glass folios' (Server returned malformed data)
[5/240] Querying Getty for: Bamboo
Failed to parse JSON for 'Bamboo' (Server returned malformed data)
[6/240] Querying Getty for: Baroque pearl
Failed to parse JSON for 'Baroque pearl' (Server returned malformed data)
[7/240] Querying Getty for: Basil
Failed to parse JSON for 'Basil' (Server returned malformed data)
[8/240] Querying Getty for: Basil ink
Failed to parse JSON for 'Basil ink' (Server returned malformed data)
[9/240] Querying Getty for: Bidri alloy i

KeyboardInterrupt: 